In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from catboost import CatBoostRegressor

In [4]:
train = pd.read_csv("../MLOpsedian/data/processed/train_model_ready.csv")
test = pd.read_csv("../MLOpsedian/data/processed/test_model_ready.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (19700, 69)
Test shape: (5300, 68)


In [5]:
def add_alpha_pack_features(df):
    df = df.copy()
    eps = 1e-5
    
    # Use clipped columns if available because raw distance/rainfall had invalid negative values
    if "distance_to_river_m_clipped" in df.columns:
        distance = df["distance_to_river_m_clipped"]
    else:
        distance = df["distance_to_river_m"].clip(lower=0)
    
    if "rainfall_7d_mm_clipped" in df.columns:
        rainfall_7d = df["rainfall_7d_mm_clipped"]
    else:
        rainfall_7d = df["rainfall_7d_mm"].clip(lower=0)
    
    inundation = df["inundation_area_sqm"].clip(lower=0)
    
    # 1. The Golden Ratio
    df["GOLDEN_distance_rainfall_ratio"] = (
        np.log1p(distance) - np.log1p(rainfall_7d + eps)
    )
    
    # 2. Flood Spread Ratio
    df["distance_to_river_DIV_inundation_area"] = (
        distance / (inundation + eps)
    )
    
    # 3. Water Velocity Proxy
    df["distance_to_river_DIV_rainfall_7d"] = (
        distance / (rainfall_7d + eps)
    )
    
    # 4. Water Volume Proxy
    df["rainfall_7d_MULT_inundation_area"] = (
        rainfall_7d * inundation
    )
    
    # Safety check: replace any accidental infinite values
    new_cols = [
        "GOLDEN_distance_rainfall_ratio",
        "distance_to_river_DIV_inundation_area",
        "distance_to_river_DIV_rainfall_7d",
        "rainfall_7d_MULT_inundation_area"
    ]
    
    for col in new_cols:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    return df

In [6]:
train_fe = add_alpha_pack_features(train)
test_fe = add_alpha_pack_features(test)

print("Train shape after feature engineering:", train_fe.shape)
print("Test shape after feature engineering:", test_fe.shape)

Train shape after feature engineering: (19700, 73)
Test shape after feature engineering: (5300, 72)


In [7]:
alpha_pack_features = [
    "district",
    "distance_to_river_DIV_inundation_area",
    "distance_to_river_DIV_rainfall_7d",
    "GOLDEN_distance_rainfall_ratio",
    "generation_date",
    "reason_not_good_to_live",
    "inundation_area_sqm",
    "infrastructure_score",
    "extreme_weather_index",
    "terrain_roughness_index",
    "road_quality",
    "monthly_rainfall_mm_log1p",
    "landcover",
    "rainfall_7d_MULT_inundation_area",
    "place_name",
    "rainfall_7d_mm",
    "seasonal_index",
    "ndwi_qmap",
    "latitude",
    "longitude",
    "distance_to_river_m",
    "ndwi",
    "water_supply",
    "nearest_evac_km_log1p",
    "elevation_m_yeojohnson",
    "monthly_rainfall_mm",
    "socioeconomic_status_index",
    "population_density_per_km2_log1p",
    "water_presence_flag",
    "nearest_hospital_km_log1p",
    "ndvi_qmap",
    "flood_occurrence_current_event",
    "rainfall_7d_mm_log1p",
    "soil_type",
    "population_density_per_km2"
]

print("Number of alpha pack features:", len(alpha_pack_features))

Number of alpha pack features: 35


In [8]:
missing_train_features = [
    col for col in alpha_pack_features
    if col not in train_fe.columns
]

missing_test_features = [
    col for col in alpha_pack_features
    if col not in test_fe.columns
]

print("Missing features in train:", missing_train_features)
print("Missing features in test:", missing_test_features)

Missing features in train: []
Missing features in test: []


In [9]:
id_col = "record_id"
target = "flood_risk_score"

X = train_fe[alpha_pack_features].copy()
y = train_fe[target].copy()

X_test = test_fe[alpha_pack_features].copy()
test_ids = test_fe[id_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("Missing in X:", X.isnull().sum().sum())
print("Missing in X_test:", X_test.isnull().sum().sum())

print("Feature columns match:", list(X.columns) == list(X_test.columns))

X shape: (19700, 35)
y shape: (19700,)
X_test shape: (5300, 35)
Missing in X: 0
Missing in X_test: 0
Feature columns match: True


In [10]:
cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("Number of categorical features:", len(cat_features))
print(cat_features)

Number of categorical features: 10
['district', 'generation_date', 'reason_not_good_to_live', 'road_quality', 'landcover', 'place_name', 'water_supply', 'water_presence_flag', 'flood_occurrence_current_event', 'soil_type']


In [11]:
target_bins = pd.cut(
    y,
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.001],
    labels=[0, 1, 2, 3, 4]
)

print("Target bin distribution:")
print(target_bins.value_counts().sort_index())

Target bin distribution:
flood_risk_score
0    2390
1    4402
2    7650
3    3353
4    1905
Name: count, dtype: int64


In [12]:
def validate_submission(submission):
    print("Submission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

In [13]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, target_bins), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / skf.n_splits

print("\n" + "=" * 70)
print("CatBoost Alpha Pack CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))


Fold 1
0:	learn: 0.2355842	test: 0.2356074	best: 0.2356074 (0)	total: 85ms	remaining: 2m 49s
200:	learn: 0.2280342	test: 0.2314374	best: 0.2314374 (200)	total: 2.01s	remaining: 18s
400:	learn: 0.2241699	test: 0.2313742	best: 0.2313062 (310)	total: 4.04s	remaining: 16.1s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2313062088
bestIteration = 310

Shrink model to first 311 iterations.
Fold RMSE: 0.2313062088784837
Fold MAE : 0.1769832796789167
Fold R²  : 0.037575487708312316

Fold 2
0:	learn: 0.2351942	test: 0.2371106	best: 0.2371106 (0)	total: 12.5ms	remaining: 24.9s
200:	learn: 0.2273923	test: 0.2334285	best: 0.2334188 (193)	total: 4.36s	remaining: 39s
400:	learn: 0.2239065	test: 0.2334115	best: 0.2333816 (266)	total: 9.26s	remaining: 36.9s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.2333815548
bestIteration = 266

Shrink model to first 267 iterations.
Fold RMSE: 0.23338155496456145
Fold MAE : 0.17756137181459383
Fold R²  : 0.0326568613

In [14]:
alpha_pack_preds = np.clip(test_preds, 0, 1)

print("Prediction min:", alpha_pack_preds.min())
print("Prediction max:", alpha_pack_preds.max())
print("Prediction mean:", alpha_pack_preds.mean())
print("Prediction std:", alpha_pack_preds.std())

sub_alpha_pack = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": alpha_pack_preds
})

validate_submission(sub_alpha_pack)

sub_alpha_pack.to_csv(
    "../submissions/sub_v008_catboost_alpha_pack.csv",
    index=False
)

print("Saved: ../submissions/sub_v008_catboost_alpha_pack.csv")

Prediction min: 0.35899341173190963
Prediction max: 0.6241741485433789
Prediction mean: 0.47785288208785803
Prediction std: 0.04223845327527326
Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.473327
1   F100765          0.443697
2   F107573          0.488596
3   F110345          0.547429
4   F118850          0.503854

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.35899341173190963
Max: 0.6241741485433789

Submission is valid.
Saved: ../submissions/sub_v008_catboost_alpha_pack.csv


In [15]:
train_mean = y.mean()

alpha_pack_preds_conservative_095 = (
    0.95 * alpha_pack_preds + 0.05 * train_mean
)

alpha_pack_preds_conservative_095 = np.clip(
    alpha_pack_preds_conservative_095,
    0,
    1
)

sub_alpha_pack_conservative = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": alpha_pack_preds_conservative_095
})

validate_submission(sub_alpha_pack_conservative)

sub_alpha_pack_conservative.to_csv(
    "../submissions/sub_v009_catboost_alpha_pack_conservative_095.csv",
    index=False
)

print("Saved: ../submissions/sub_v009_catboost_alpha_pack_conservative_095.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.473587
1   F100765          0.445438
2   F107573          0.488092
3   F110345          0.543983
4   F118850          0.502588

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.3649696123382075
Max: 0.6168913123091033

Submission is valid.
Saved: ../submissions/sub_v009_catboost_alpha_pack_conservative_095.csv


In [16]:
comparison = pd.DataFrame({
    "alpha_pack": alpha_pack_preds,
    "alpha_pack_conservative_095": alpha_pack_preds_conservative_095
})

comparison.describe()

,alpha_pack,alpha_pack_conservative_095
count,5300.000000,5300.000000
mean,0.477853,0.477886
std,0.042242,0.040130
min,0.358993,0.364970
25%,0.446531,0.448130
50%,0.475600,0.475746
75%,0.507788,0.506324
max,0.624174,0.616891


In [17]:
print("Alpha pack 0 predictions:", (alpha_pack_preds == 0).sum())
print("Alpha pack 1 predictions:", (alpha_pack_preds == 1).sum())

print("Conservative alpha pack 0 predictions:", (alpha_pack_preds_conservative_095 == 0).sum())
print("Conservative alpha pack 1 predictions:", (alpha_pack_preds_conservative_095 == 1).sum())

Alpha pack 0 predictions: 0
Alpha pack 1 predictions: 0
Conservative alpha pack 0 predictions: 0
Conservative alpha pack 1 predictions: 0


In [18]:
catboost_comparison = pd.DataFrame([
    {
        "version": "sub_v006_catboost_baseline",
        "model": "CatBoost Baseline",
        "features": "66 cleaned features",
        "validation": "Normal KFold",
        "cv_rmse": 0.23176890534936953,
        "cv_mae": 0.17695452528072222,
        "cv_r2": 0.033350653634844754,
        "public_score": 0.38368
    },
    {
        "version": "sub_v008_catboost_alpha_pack",
        "model": "CatBoost Alpha Pack",
        "features": "35 selected + engineered features",
        "validation": "Target-binned StratifiedKFold",
        "cv_rmse": 0.23164778640435246,
        "cv_mae": 0.17673736118136243,
        "cv_r2": 0.03463008143352864,
        "public_score": None
    }
])

catboost_comparison["rmse_vs_baseline"] = (
    catboost_comparison["cv_rmse"] - catboost_comparison.loc[0, "cv_rmse"]
)

catboost_comparison["mae_vs_baseline"] = (
    catboost_comparison["cv_mae"] - catboost_comparison.loc[0, "cv_mae"]
)

catboost_comparison["r2_vs_baseline"] = (
    catboost_comparison["cv_r2"] - catboost_comparison.loc[0, "cv_r2"]
)

catboost_comparison.to_csv(
    "../reports/catboost_model_comparison.csv",
    index=False
)

catboost_comparison

,version,model,features,validation,cv_rmse,cv_mae,cv_r2,public_score,rmse_vs_baseline,mae_vs_baseline,r2_vs_baseline
0,sub_v006_catboost_baseline,CatBoost Baseline,66 cleaned features,Normal KFold,0.231769,0.176955,0.033351,0.38368,0.000000,0.000000,0.000000
1,sub_v008_catboost_alpha_pack,CatBoost Alpha Pack,35 selected + engineered features,Target-binned StratifiedKFold,0.231648,0.176737,0.034630,NaN,-0.000121,-0.000217,0.001279
